# HireLens — Fine-tune BERT (Mohler + data tự thu)

**Mục tiêu:** regression điểm 0–10 từ câu hỏi + câu trả lời (+ đáp án mẫu).

| Data | Vai trò |
|------|--------|
| Mohler ASAG (`nkazi/MohlerASAG`, raw/open_ended) | Benchmark CS, điểm 0–5 → ×2 thành 0–10 |
| `AI/data/answers.csv` | Domain phỏng vấn Spring Boot (tự thu) |

## Cách chạy trên Google Colab

1. Runtime → Change runtime type → **T4 GPU** (hoặc GPU bất kỳ).
2. Upload folder `AI/` lên Drive **hoặc** zip rồi upload vào `/content`.
3. Chạy lần lượt các cell.
4. Nếu HF yêu cầu login (dataset gated): chạy cell `huggingface-cli login`.

**Scenario khuyến nghị báo cáo:** `combined` hoặc `transfer`.

## 0) Mount Drive / vào thư mục AI

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Sửa đường dẫn cho đúng chỗ bạn upload folder AI
AI_DIR = '/content/drive/MyDrive/recruit_system/AI'

import os
assert os.path.isdir(AI_DIR), f'Không thấy {AI_DIR} — sửa AI_DIR'
os.chdir(AI_DIR)
!pwd
!ls data output src | head

## 1) Cài thư viện

In [ ]:
!pip install -q datasets transformers accelerate scikit-learn pandas numpy tqdm sentencepiece
# torch thường có sẵn trên Colab GPU
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 2) (Nếu cần) Login Hugging Face

Dataset `nkazi/MohlerASAG` có thể yêu cầu chấp nhận điều khoản trên trang HF rồi login token.

In [ ]:
# Bỏ comment nếu load_dataset báo gated / 401
# from huggingface_hub import login
# login()  # dán HF token (read)

from datasets import load_dataset
ds = load_dataset('nkazi/MohlerASAG', 'raw', split='open_ended')
print(ds)
print(ds[0])

## 3) Fine-tune

- `combined`: trộn Mohler train + self train, đánh giá riêng trên từng test set  
- `transfer`: pretrain Mohler → fine-tune self (khuyến nghị nếu muốn “kết hợp” rõ ràng)  
- `mohler` / `self`: train riêng từng nguồn

In [ ]:
SCENARIO = 'combined'  # combined | transfer | mohler | self

import subprocess, sys
cmd = [
    sys.executable, '-m', 'src.finetune_bert',
    '--scenario', SCENARIO,
    '--model-name', 'bert-base-multilingual-cased',
    '--epochs', '3',
    '--mohler-epochs', '2',
    '--batch-size', '8',
    '--lr', '2e-5',
    '--max-length', '256',
]
print(' '.join(cmd))
subprocess.check_call(cmd)

## 4) Xem metrics (đưa vào báo cáo)

In [ ]:
import json
from pathlib import Path

out = Path('output') / f'bert_finetuned_{SCENARIO}'
meta = json.loads((out / 'finetune_metrics.json').read_text(encoding='utf-8'))
print(json.dumps(meta['metrics'], ensure_ascii=False, indent=2))
print('Model dir:', out / 'model')

## 5) Thử chấm 1 câu (self domain)

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_dir = f'output/bert_finetuned_{SCENARIO}/model'
tok = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
model.eval()

question = 'Spring Boot là gì? Nêu ưu điểm chính so với Spring Framework truyền thống.'
reference = 'Spring Boot dựa trên Spring, auto-configuration, starter, embedded server, REST/API.'
answer = 'Spring Boot dựa trên Spring, có auto-configuration và starter, hay dùng REST API.'
text = f'Question: {question}\nReference: {reference}\nAnswer: {answer}'

inputs = tok(text, return_tensors='pt', truncation=True, max_length=256)
with torch.no_grad():
    score = model(**inputs).logits.item()
print('predicted_score (0-10):', round(max(0, min(10, score)), 2))

## Gợi ý viết báo cáo

- Bảng S1: metric trên `self_test` (domain đồ án)
- Bảng S2: metric trên `mohler_test` (benchmark)
- So với `self_test_baseline_f1x10` trong cùng file JSON
- Nêu: Mohler thang gốc 0–5 đã nhân 2; self giữ 0–10